In [3]:
from pyspark.sql import functions as F

production_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("Files/bronze/production/production.csv")
)

print("Bronze rows:", production_df.count())
display(production_df.limit(10))

StatementMeta(, 36f47133-bb3d-48dd-9da3-6fbead5b772a, 5, Finished, Available, Finished, False)

Bronze rows: 50010


SynapseWidget(Synapse.DataFrame, a1e129bd-9b0b-42a3-b351-22d889d37e28)

In [4]:
# Remove exact duplicates
df = production_df.dropDuplicates()

# Cast numeric columns
df = (
    df
    .withColumn("planned_quantity", F.col("planned_quantity").cast("int"))
    .withColumn("actual_quantity", F.col("actual_quantity").cast("int"))
    .withColumn("downtime_minutes", F.col("downtime_minutes").cast("int"))
)

# Remove records with missing critical keys
df = df.dropna(subset=["production_id", "machine_id", "product_id"])

# Remove invalid machine IDs
df = df.filter(F.col("machine_id").isin("M001", "M002", "M003", "M004", "M005"))

# Remove invalid product IDs
df = df.filter(F.col("product_id").isin("P001", "P002", "P003", "P004", "P005"))

# Remove invalid quantity
df = df.filter(
    (F.col("planned_quantity") >= 0) &
    (F.col("actual_quantity") >= 0)
)

# Keep valid production statuses
df = df.filter(
    F.col("production_status").isin(
        "Completed",
        "In Progress",
        "Delayed"
    )
)

print("Silver rows:", df.count())
display(df)

StatementMeta(, 36f47133-bb3d-48dd-9da3-6fbead5b772a, 6, Finished, Available, Finished, False)

Silver rows: 39933


SynapseWidget(Synapse.DataFrame, 88f4ac0b-1d31-4a33-a499-a0203ef3634c)

In [5]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver_production")
)

StatementMeta(, 36f47133-bb3d-48dd-9da3-6fbead5b772a, 7, Finished, Available, Finished, False)

In [6]:
silver_production = spark.table("silver_production")

print("Silver rows:", silver_production.count())
silver_production.printSchema()

display(silver_production.limit(10))

StatementMeta(, 36f47133-bb3d-48dd-9da3-6fbead5b772a, 8, Finished, Available, Finished, False)

Silver rows: 39933
root
 |-- production_id: string (nullable = true)
 |-- production_date: timestamp (nullable = true)
 |-- machine_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- planned_quantity: integer (nullable = true)
 |-- actual_quantity: integer (nullable = true)
 |-- downtime_minutes: integer (nullable = true)
 |-- production_status: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 662429fd-ac57-4c3c-9778-bb5fe1e4e61a)

In [7]:
machine_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("Files/bronze/machine/machine.csv")
    .dropDuplicates(["machine_id"])
)

product_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("Files/bronze/product/product.csv")
    .dropDuplicates(["product_id"])
)

machine_df.write.format("delta").mode("overwrite").saveAsTable("silver_machine")
product_df.write.format("delta").mode("overwrite").saveAsTable("silver_product")

print("Machine:", machine_df.count())
print("Product:", product_df.count())

StatementMeta(, 36f47133-bb3d-48dd-9da3-6fbead5b772a, 9, Finished, Available, Finished, False)

Machine: 5
Product: 5


In [8]:
maintenance_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("Files/bronze/maintenance/maintenance.csv")
    .dropDuplicates(["maintenance_id"])
)

maintenance_df = (
    maintenance_df
    .withColumn("downtime_hours", F.col("downtime_hours").cast("double"))
    .withColumn("maintenance_cost", F.col("maintenance_cost").cast("double"))
    .filter(F.col("machine_id").isin("M001", "M002", "M003", "M004", "M005"))
    .filter(F.col("downtime_hours") >= 0)
    .filter(F.col("maintenance_cost") >= 0)
)

maintenance_df.write.format("delta").mode("overwrite").saveAsTable("silver_maintenance")

print("Maintenance:", maintenance_df.count())

StatementMeta(, 36f47133-bb3d-48dd-9da3-6fbead5b772a, 10, Finished, Available, Finished, False)

Maintenance: 5000


In [9]:
quality_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("Files/bronze/quality_inspection/quality_inspection.csv")
    .dropDuplicates(["inspection_id"])
)

quality_df = (
    quality_df
    .withColumn("inspected_quantity", F.col("inspected_quantity").cast("int"))
    .withColumn("defect_quantity", F.col("defect_quantity").cast("int"))
    .dropna(subset=["inspection_id", "production_id", "machine_id", "product_id"])
    .filter(F.col("machine_id").isin("M001", "M002", "M003", "M004", "M005"))
    .filter(F.col("product_id").isin("P001", "P002", "P003", "P004", "P005"))
    .filter(F.col("inspected_quantity") >= 0)
    .filter(F.col("defect_quantity") >= 0)
    .filter(F.col("defect_quantity") <= F.col("inspected_quantity"))
    .filter(F.col("quality_status").isin("Passed", "Failed"))
)

quality_df.write.format("delta").mode("overwrite").saveAsTable("silver_quality_inspection")

print("Quality:", quality_df.count())

StatementMeta(, 36f47133-bb3d-48dd-9da3-6fbead5b772a, 11, Finished, Available, Finished, False)

Quality: 39960


In [10]:
spark.sql("SHOW TABLES").show()

StatementMeta(, 36f47133-bb3d-48dd-9da3-6fbead5b772a, 12, Finished, Available, Finished, False)

+--------------------+--------------------+-----------+
|           namespace|           tableName|isTemporary|
+--------------------+--------------------+-----------+
|`Industrial Analy...|      silver_machine|      false|
|`Industrial Analy...|  silver_maintenance|      false|
|`Industrial Analy...|      silver_product|      false|
|`Industrial Analy...|   silver_production|      false|
|`Industrial Analy...|silver_quality_in...|      false|
+--------------------+--------------------+-----------+

